# 파이프라인
파이프라인은 전처리와 모델링 과정을 하나의 모델처럼 묶는 기능입니다.



In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR # SVM Regression

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

datas = fetch_ucirepo(id=9)

In [10]:
df = pd.concat((
  datas.data.features,
  datas.data.targets
), axis=1)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   displacement  398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   horsepower    392 non-null    float64
 3   weight        398 non-null    int64  
 4   acceleration  398 non-null    float64
 5   model_year    398 non-null    int64  
 6   origin        398 non-null    int64  
 7   mpg           398 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 25.0 KB


In [11]:
df = df.dropna(axis=0)
df.shape

(392, 8)

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
  df[df.drop(columns=["mpg"]).columns], df["mpg"],
  test_size=0.2,
  random_state=42
)

In [ ]:
pipe =  Pipeline([
  ("scaler", StandardScaler()), # 스케일러 지정
  ("svr", SVR()) # 사용할 모델 지정
])

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)

In [39]:
y_test_nor = y_test.reset_index(drop=True)
type(y_test_nor)

pandas.Series

In [51]:
pred_df = pd.DataFrame(pred, columns=["pred"])
result_df = pd.concat((
  pred_df,
  y_test.reset_index(drop=True),
  pd.DataFrame((pred_df['pred']-y_test_nor), columns=["diff"])
), axis=1)

result_df.apply(
  lambda x: pd.Series({
    "mean": pd.Series.mean(abs(x)),
    "std": abs(x).std(),
    "min": min(abs(x)),
    "max": max(abs(x))
  })
)

,pred,mpg,diff
mean,23.066066,22.837975,2.117410
std,6.315700,7.189920,2.202686
min,13.208756,10.000000,0.035198
max,34.196106,44.000000,14.562037
